In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
%%capture
!pip install datasets accelerate bert-score pyarrow

## **01. Read data - Config**

In [76]:
from pathlib import Path
import sys
import json
import time

import numpy as np
import pandas as pd
import torch

In [77]:
# Dung model XLM-R
MODEL_NAME = "google/gemma-3-270m-it"
RUN_NAME = "gemma-3-270m-it"

ENABLE_THINKING = None

In [78]:
PASSAGE_COLUMN = 'embedding_text'      # cột corpus mà generator đọc
TARGET_COLUMN = 'abstractive answer'   # câu trả lời chuẩn
CONTEXT_SEPARATOR = '\n\n'             # chỉ dùng khi ghép nhiều passage lúc eval

NUMBER_OF_EPOCHS = 5
LEARNING_RATE = 5e-5
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CHECKPOINTING = True
WARMUP_RATIO = 0.1
RANDOM_SEED = 42

MAX_SOURCE_LENGTH = 2048
MAX_TARGET_LENGTH = 1024

GENERATION_BATCH_SIZE = 32
NUM_BEAMS = 1

TOKENIZE_KWARGS = dict(lowercase=True, strip_punctuation=True)

USE_BERTSCORE = True
BERTSCORE_MODEL = None

In [79]:
BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"

GENERATOR_DIR = BASE_DIR / "03_Generator"
OUTPUT_DIR = GENERATOR_DIR / "output"
CHECKPOINT_DIR = GENERATOR_DIR / "checkpoints" / RUN_NAME
MODEL_SAVE_DIR = GENERATOR_DIR / "models" / RUN_NAME

for directory in [OUTPUT_DIR, CHECKPOINT_DIR, MODEL_SAVE_DIR]:
  directory.mkdir(parents=True, exist_ok=True)

sys.path.append(str(GENERATOR_DIR))

In [80]:
sys.path.append(str(GENERATOR_DIR))

import generator_common
import importlib

importlib.reload(generator_common)

gc = generator_common

gc.require_version(5)

torch.manual_seed(RANDOM_SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

generator_common v5 (need >= 5)
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


### **1.2. Read data**

In [81]:
qa_master = pd.read_parquet(DATA_DIR / 'corpus' / 'qa_master.parquet')
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')

print(f"qa_master : {len(qa_master):,} cặp QA")
print(f"corpus    : {len(corpus):,} passage")

qa_master : 9,609 cặp QA
corpus    : 294 passage


In [82]:
data = {
    split_name: gc.build_generator_dataframe(qa_master, split_name, corpus, passage_column=PASSAGE_COLUMN, target_column=TARGET_COLUMN)
    for split_name in ['train', 'val', 'test']
}

for split_name, frame in data.items():
  print(f"{split_name}: {len(frame):,} mẫu, {frame['query_id'].nunique():,} câu hỏi, {frame['doc_id'].nunique():,} điều luật")


train: 7,718 mẫu, 7,697 câu hỏi, 294 điều luật
val: 933 mẫu, 933 câu hỏi, 256 điều luật
test: 958 mẫu, 958 câu hỏi, 255 điều luật


## **02. Model & Tokenizer**

In [83]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import notebook_login

# Log in to Hugging Face to access gated models
notebook_login()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)

if torch.cuda.is_available():
  model = model.cuda()

print("Model:", MODEL_NAME)
print(f"Tham số: {sum(p.numel() for p in model.parameters()):,}")
print("pad_token:", repr(tokenizer.pad_token))

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model: google/gemma-3-270m-it
Tham số: 268,098,176
pad_token: '<pad>'


## **03. Prompt**

In [84]:
PROMPT_CONFIG = {
    'system_prompt': ("Bạn hãy trả lời câu hỏi về quy định đào tạo đại học dựa trên điều luật được cung cấp. Yêu cầu: chỉ dùng thông tin trong điều luật, không thêm thông tin mới và trả lời ngắn gọn bằng một câu hoàn chỉnh."),
    'user_template': "Câu hỏi: {question}\nĐiều luật: {context}",
    'enable_thinking': ENABLE_THINKING,
    'passage_column': PASSAGE_COLUMN,
    'target_column': TARGET_COLUMN,
    'context_separator': CONTEXT_SEPARATOR,
    'max_source_length': MAX_SOURCE_LENGTH,
    'max_target_length': MAX_TARGET_LENGTH,
    'num_beams': NUM_BEAMS,
    'do_sample': False,
    'model_name': MODEL_NAME,
    'trained_with_top_k': 1,     # fine-tune với đúng MỘT điều luật
}

In [85]:
def build_user_message(question: str, context: str) -> str:
  return PROMPT_CONFIG['user_template'].format(question=str(question).strip(), context=str(context).strip())

def render_prompt_from_parts(question: str, context: str) -> str:
  messages = [
      {'role': 'system', 'content': PROMPT_CONFIG['system_prompt']},
      {'role': 'user', 'content': build_user_message(question, context)},
  ]

  template_kwargs = dict(tokenize=False, add_generation_prompt=True)

  if PROMPT_CONFIG['enable_thinking'] is not None:
    template_kwargs['enable_thinking'] = PROMPT_CONFIG['enable_thinking']

  try:
    return tokenizer.apply_chat_template(messages, **template_kwargs)
  except Exception:
    # Gemma không nhận vai trò 'system' -> gộp vào lượt user.
    merged = [{'role': 'user',
               'content': PROMPT_CONFIG['system_prompt'] + "\n\n" + build_user_message(question, context)}]

    return tokenizer.apply_chat_template(merged, **template_kwargs)



In [86]:
def render_prompt(row) -> str:
  return render_prompt_from_parts(row['source_question'], row['source_context'])

### 3.1. Test

In [87]:
import re
sample_prompt = render_prompt(data['train'].iloc[0])

sample_prompt = re.sub(r'<think>.*?</think>', ' ', sample_prompt, flags=re.DOTALL)
sample_prompt = re.sub(r'\s+', ' ', sample_prompt).strip()

print(sample_prompt[:800], "...")

if PROMPT_CONFIG['enable_thinking'] is False:
  assert '<think>' not in sample_prompt, ("Chat template vẫn chèn <think> — kiểm tra lại enable_thinking. Nếu không tắt được thì phải lọc khối <think>...</think> khỏi output trước khi tính chỉ số.")

assert data['train'].iloc[0]['source_question'][:20] in sample_prompt
assert data['train'].iloc[0]['source_context'][:30] in sample_prompt

# Kiểm tra trên TOÀN BỘ train — mẫu dài nhất mới là mẫu quan trọng.
all_prompt_lengths = [len(tokenizer.encode(render_prompt(row), add_special_tokens=False)) for row in data['train'].to_dict('records')]

all_target_lengths = [len(tokenizer.encode(target, add_special_tokens=False)) for target in data['train']['target']]

print(f"\nprompt dài nhất: {max(all_prompt_lengths)} / MAX_SOURCE_LENGTH = {MAX_SOURCE_LENGTH}")
print(f"target dài nhất: {max(all_target_lengths)} / MAX_TARGET_LENGTH = {MAX_TARGET_LENGTH}")

assert sum(1 for length in all_prompt_lengths if length > MAX_SOURCE_LENGTH) == 0, ("Có prompt bị cắt cụt. Padding là động nên tăng trần gần như không tốn thêm bộ nhớ.")
assert sum(1 for length in all_target_lengths if length > MAX_TARGET_LENGTH) == 0

print("\nSmoke test đã pass.")

<bos><start_of_turn>user Bạn hãy trả lời câu hỏi về quy định đào tạo đại học dựa trên điều luật được cung cấp. Yêu cầu: chỉ dùng thông tin trong điều luật, không thêm thông tin mới và trả lời ngắn gọn bằng một câu hoàn chỉnh. Câu hỏi: Sinh viên dự bị không trở thành sinh viên chính thức bao nhiêu học kỳ sẽ bị loại khỏi CTTN? Điều luật: Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG Điều 9. Tuyển bổ sung và loại ra khỏi chương trình, xét chính thức và dự bị Đối tượng tham gia CTTN là những sinh viên có năng lực xuất sắc, do đó, sau mỗi học kỳ BĐH quyết định việc loại sinh viên khỏi lớp tài năng, tuyển bổ sung sinh viên từ chương trình chuẩn vào lớp tài năng, xét chuyển đổi sinh viên chính thức và dự bị. Đầu mỗi học kỳ, Khoa xét và đề nghị lên BĐH các danh sách sinh viên tuyển bổ sung, bị l ...

prompt dài nhất: 1884 / MAX_SOURCE_LENGTH = 2048
target dài nhất: 990 / MAX_TARGET_LENGTH = 1024

Smoke test đã pass.


## **04. Dataset + mask loss**

In [88]:
from torch.utils.data import Dataset

class GeneratorDataset(Dataset):
  """Nối prompt với target, đặt label = -100 cho toàn bộ token thuộc prompt."""

  def __init__(self, frame: pd.DataFrame):
    self.examples = []

    for row in frame.to_dict('records'):
      prompt = render_prompt(row)

      # Cắt giữ ĐUÔI. Lúc sinh phải đặt tokenizer.truncation_side='left' cho khớp.
      prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)[-MAX_SOURCE_LENGTH:]

      target_ids = tokenizer.encode(row['target'], add_special_tokens=False)[:MAX_TARGET_LENGTH]
      target_ids = target_ids + [tokenizer.eos_token_id]

      self.examples.append({
          'input_ids': prompt_ids + target_ids,
          'labels': [-100] * len(prompt_ids) + target_ids,
      })

  def __len__(self):
    return len(self.examples)

  def __getitem__(self, index):
    return self.examples[index]

In [89]:
def collate(batch):
  max_length = max(len(item['input_ids']) for item in batch)

  input_ids, labels, attention_mask = [], [], []

  for item in batch:
    padding = max_length - len(item['input_ids'])

    # Pad BÊN PHẢI khi huấn luyện (lúc sinh mới cần pad bên trái).
    input_ids.append(item['input_ids'] + [tokenizer.pad_token_id] * padding)
    labels.append(item['labels'] + [-100] * padding)
    attention_mask.append([1] * len(item['input_ids']) + [0] * padding)

  return {
      'input_ids': torch.tensor(input_ids),
      'labels': torch.tensor(labels),
      'attention_mask': torch.tensor(attention_mask),
  }

In [90]:
train_dataset = GeneratorDataset(data['train'])
validation_dataset = GeneratorDataset(data['val'])

print(f"train: {len(train_dataset):,} | val: {len(validation_dataset):,}")

train: 7,718 | val: 933


In [91]:
# Kiểm tra mask: chỉ phần target mới có label khác -100.
example = train_dataset[0]

masked_count = sum(1 for label in example['labels'] if label == -100)
kept_count = len(example['labels']) - masked_count

decoded_target = tokenizer.decode(
    [label for label in example['labels'] if label != -100], skip_special_tokens=True
)

print(f"Token bị mask (prompt): {masked_count} | token tính loss (target): {kept_count}")
print("Phần tính loss giải mã ra:", repr(decoded_target))

assert kept_count > 0, "Không có token nào được tính loss."
assert data['train'].iloc[0]['target'][:20] in decoded_target, "Mask bị lệch."

print("\nMask loss đúng.")

Token bị mask (prompt): 1180 | token tính loss (target): 27
Phần tính loss giải mã ra: 'Sinh viên dự bị không trở thành sinh viên chính thức 02 học kỳ liên tiếp sẽ bị loại khỏi CTTN.'

Mask loss đúng.


## **05. Training**

In [92]:
from transformers import Trainer, TrainingArguments
import math

steps_per_epoch = math.ceil(len(train_dataset) / (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))

warmup_steps = int(steps_per_epoch * NUMBER_OF_EPOCHS * WARMUP_RATIO)

training_arguments = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUMBER_OF_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={'use_reentrant': False} if GRADIENT_CHECKPOINTING else None,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    seed=RANDOM_SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collate,
)

start_time = time.perf_counter()

trainer.train()

training_seconds = time.perf_counter() - start_time

print(f"\nHuấn luyện xong trong {training_seconds / 60:.1f} phút")

model.save_pretrained(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

print("Đã lưu:", MODEL_SAVE_DIR)

Epoch,Training Loss,Validation Loss
1,0.288891,0.300973
2,0.246085,0.269215
3,0.165076,0.270943
4,0.095754,0.309821
5,0.056600,0.358621


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



Huấn luyện xong trong 44.7 phút


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu: /content/drive/MyDrive/Legal_Sytem/03_Generator/models/gemma-3-270m-it


## **06. Generation**

In [93]:
def clean_generation(text: str) -> str:
  """Lọc khối <think> nếu model vẫn sinh ra, và bỏ khoảng trắng thừa."""
  text = re.sub(r'<think>.*?</think>', ' ', text, flags=re.DOTALL)
  text = re.sub(r'\s+', ' ', text)

  return text.strip()

In [94]:
@torch.no_grad()
def generate_answers(frame: pd.DataFrame, batch_size: int = GENERATION_BATCH_SIZE) -> list[str]:
  model.eval()

  original_padding_side = tokenizer.padding_side
  original_truncation_side = tokenizer.truncation_side

  tokenizer.padding_side = 'left'       # bắt buộc khi sinh theo batch
  tokenizer.truncation_side = 'left'    # khớp prompt_ids[-MAX_SOURCE_LENGTH:] lúc train

  prompts = [render_prompt(row) for row in frame.to_dict('records')]

  outputs = []

  for start in range(0, len(prompts), batch_size):
    encoded = tokenizer(prompts[start:start + batch_size], return_tensors='pt', padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH, add_special_tokens=False)

    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    generated = model.generate(**encoded, max_new_tokens=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

    new_tokens = generated[:, encoded['input_ids'].shape[1]:]

    outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

  tokenizer.padding_side = original_padding_side
  tokenizer.truncation_side = original_truncation_side

  return [clean_generation(text) for text in outputs]

In [95]:
predictions_by_split = {}
per_example_by_split = {}
summary_by_split = {}

for split_name in ['val', 'test']:
  frame = data[split_name]

  start_time = time.perf_counter()

  hypotheses = generate_answers(frame)

  generation_seconds = time.perf_counter() - start_time

  references = frame['target'].tolist()

  per_example = gc.evaluate_generations(references, hypotheses, frame['qa_id'].tolist(), use_bertscore=USE_BERTSCORE, bertscore_model=BERTSCORE_MODEL, **TOKENIZE_KWARGS)

  summary = gc.summarize_generations(references, hypotheses, per_example, **TOKENIZE_KWARGS)
  summary['latency_ms_per_example'] = round(generation_seconds / len(frame) * 1000, 2)

  predictions_by_split[split_name] = frame.assign(prediction=hypotheses)
  per_example_by_split[split_name] = per_example
  summary_by_split[split_name] = summary

  print(f"[{split_name}] " + "  ".join(f"{key}={value}" for key, value in summary.items()))

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

[val] BLEU1=66.82  BLEU4=60.66  ROUGE-L=69.45  F1=73.05  BERTScore=89.23999786376953  latency_ms_per_example=440.32


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[test] BLEU1=62.24  BLEU4=55.12  ROUGE-L=65.29  F1=69.97  BERTScore=88.18000030517578  latency_ms_per_example=452.86


### **6.1. Compare to baseline**

In [96]:
comparison_records = []

for split_name in ['val', 'test']:
  frame = data[split_name]
  references = frame['target'].tolist()

  for baseline_name, baseline_hypotheses in gc.trivial_baselines(frame).items():
    baseline_per_example = gc.evaluate_generations(references, baseline_hypotheses, frame['qa_id'].tolist(), use_bertscore=False, **TOKENIZE_KWARGS)

    comparison_records.append({
        'split': split_name, 'hệ_thống': baseline_name,
        **gc.summarize_generations(references, baseline_hypotheses,
                                    baseline_per_example, **TOKENIZE_KWARGS),
    })

  comparison_records.append({
      'split': split_name, 'hệ_thống': RUN_NAME,
      **{key: value for key, value in summary_by_split[split_name].items()
         if key != 'latency_ms_per_example'},
  })

display(pd.DataFrame(comparison_records))

,split,hệ_thống,BLEU1,BLEU4,ROUGE-L,F1,BERTScore
0,val,copy_question,23.99,20.75,54.66,59.45,NaN
1,val,copy_context,9.66,8.29,22.60,21.69,NaN
2,val,gemma-3-270m-it,66.82,60.66,69.45,73.05,89.239998
3,test,copy_question,27.78,23.69,54.83,60.06,NaN
4,test,copy_context,8.58,7.19,20.88,20.02,NaN
5,test,gemma-3-270m-it,62.24,55.12,65.29,69.97,88.180000


In [97]:
# Chênh lệch so với baseline mạnh nhất có ý nghĩa thống kê không? ----------
test_frame = data['test']
test_references = test_frame['target'].tolist()

strong_baseline_name = 'copy_question'
strong_baseline = gc.trivial_baselines(test_frame)[strong_baseline_name]

baseline_per_example = gc.evaluate_generations(test_references, strong_baseline, test_frame['qa_id'].tolist(), use_bertscore=USE_BERTSCORE, bertscore_model=BERTSCORE_MODEL, **TOKENIZE_KWARGS)

metrics = ['rouge_l', 'token_f1', 'sentence_bleu1'] + (['bertscore'] if USE_BERTSCORE else [])

bootstrap_records = []

for metric in metrics:
  result = gc.paired_bootstrap(per_example_by_split['test'], baseline_per_example, metric=metric)

  bootstrap_records.append({
      'metric': metric,
      RUN_NAME: round(result['mean_a'] * 100, 2),
      strong_baseline_name: round(result['mean_b'] * 100, 2),
      'chênh_lệch': round(result['difference'] * 100, 2),
      'ci_95': f"[{result['ci_lower_95'] * 100:+.2f}, {result['ci_upper_95'] * 100:+.2f}]",
      'p_value': round(result['p_value'], 4),
      'significant': result['significant_at_0.05'],
  })

display(pd.DataFrame(bootstrap_records))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,metric,gemma-3-270m-it,copy_question,chênh_lệch,ci_95,p_value,significant
0,rouge_l,65.29,54.83,10.47,"[+8.51, +12.41]",0.0,True
1,token_f1,69.97,60.06,9.91,"[+8.00, +11.81]",0.0,True
2,sentence_bleu1,62.67,46.28,16.39,"[+14.13, +18.64]",0.0,True
3,bertscore,88.18,82.16,6.02,"[+5.36, +6.67]",0.0,True


## **07. Save**

In [98]:
with open(MODEL_SAVE_DIR / 'prompt_config.json', 'w', encoding='utf-8') as file:
  json.dump(PROMPT_CONFIG, file, ensure_ascii=False, indent=2)

print("Đã lưu:", MODEL_SAVE_DIR / 'prompt_config.json')

run_config = {
    'run_name': RUN_NAME,
    'model_name': MODEL_NAME,
    'architecture': 'decoder-only',
    'model_save_dir': str(MODEL_SAVE_DIR),
    'evaluation_condition': 'oracle (gold doc_id từ qa_master)',
    'prompt_config': PROMPT_CONFIG,
    'number_of_epochs': NUMBER_OF_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'effective_batch_size': PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'gradient_checkpointing': GRADIENT_CHECKPOINTING,
    'training_method': 'full',
    'generation_batch_size': GENERATION_BATCH_SIZE,
    'tokenize_kwargs': TOKENIZE_KWARGS,
    'generator_common_version': gc.MODULE_VERSION,
    'training_examples': len(train_dataset),
    'training_minutes': round(training_seconds / 60, 2),
    'peak_vram_gb': (round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None),
}
summary_dataframe = gc.save_run(OUTPUT_DIR, RUN_NAME, predictions_by_split, per_example_by_split, summary_by_split, run_config)

display(summary_dataframe)

Đã lưu: /content/drive/MyDrive/Legal_Sytem/03_Generator/models/gemma-3-270m-it/prompt_config.json


,run_name,split,BLEU1,BLEU4,ROUGE-L,F1,BERTScore,latency_ms_per_example
0,gemma-3-270m-it,val,66.82,60.66,69.45,73.05,89.239998,440.32
1,gemma-3-270m-it,test,62.24,55.12,65.29,69.97,88.180000,452.86


In [99]:
leaderboard = gc.collect_leaderboard(OUTPUT_DIR)

display(leaderboard[leaderboard['split'] == 'test'])

,run_name,split,BLEU1,BLEU4,ROUGE-L,BERTScore,F1
0,Qwen3-0.6B,test,74.60,67.50,74.11,90.95,78.81
1,gemma-3-270m-it,test,62.24,55.12,65.29,88.18,69.97
